# Analysis 19 — who writes like whom?

Character-trigram counts on equal windows of each author's text (log: `analysis-log.md`, Analysis 19).

Choices so far (student): messages glued per author with a line break, cut into 1,500-character
windows, the same number of windows per author, links replaced by `<URL>`, capitals kept.
Stage 3 option C: raw counts first, then read back which trigrams drive the distances before
deciding whether to drop or standardise anything.

In [ ]:
import re

import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import Pipeline, TransformBase
from sklearn.feature_extraction.text import CountVectorizer

from wa_analyzer.data import load_own_chat

SEED = 42
WINDOW = 1_500  # characters per window (student)
SEPARATOR = "\n"  # between messages; Stage 6 reruns with " "
GLUED_SYSTEM_LINE = re.compile(r"\d{2}-\d{2}-\d{4} \d{2}:\d{2} - ")  # parser glitch; carries real names
NOT_A_TEXT = re.compile(r"<Media weggelaten>|bericht is verwijderd|locatie gedeeld|^locatie: |\.vcf \(bestand")
URL = re.compile(r"https?://\S+|www\.\S+")

## Clean

The lesson's `NormaliseText` lowercases and collapses all whitespace, which would undo two of the
choices above (capitals, line breaks). This step keeps both and only swaps links for a placeholder.

In [ ]:
class CleanForStyle(TransformBase):
    """Drop rows that are not typed text, and replace links with one placeholder."""

    def transform(self, data: pd.DataFrame, column: str, feature: str = "text") -> pd.DataFrame:
        text = data[column].fillna("").astype(str)
        typed = ~text.str.contains(GLUED_SYSTEM_LINE) & ~text.str.contains(NOT_A_TEXT) & (text.str.strip() != "")
        data = data[typed].copy()
        data[feature] = text[typed].str.replace(URL, "<URL>", regex=True)
        return data


own = load_own_chat().sort_values(["author", "timestamp"])
clean = Pipeline().add(CleanForStyle, column="message").apply(own)
print(f"{len(clean):,} of {len(own):,} messages kept")

## Windows

Each author's messages, in time order, glued into one text and cut at exactly `WINDOW` characters
(a window may start or end mid-message). The leftover at the end is dropped. Every author then
gets the same number of windows, drawn at random from all of theirs.

In [ ]:
def windows_per_author(frame: pd.DataFrame, size: int, separator: str) -> pd.DataFrame:
    """Cut each author's glued text into consecutive windows of exactly `size` characters."""
    rows = []
    for author, texts in frame.groupby("author").text:
        glued = separator.join(texts)
        rows += [{"author": author, "text": glued[i : i + size]} for i in range(0, len(glued) - size + 1, size)]
    return pd.DataFrame(rows)


every_window = windows_per_author(clean, WINDOW, SEPARATOR)
per_author = every_window.author.value_counts()
n_each = per_author.min()
windows = every_window.groupby("author").sample(n_each, random_state=SEED).reset_index(drop=True)
print(f"windows per author before balancing:\n{per_author.to_string()}")
print(f"\nbalanced: {n_each} windows x {windows.author.nunique()} authors = {len(windows)}")

## Counts

`lowercase=False`: `CountVectorizer` lowercases by default, which would throw the capitals away.

In [ ]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3), lowercase=False)
counts = vectorizer.fit_transform(windows.text).toarray()
trigrams = vectorizer.get_feature_names_out()
print(f"{counts.shape[0]} windows x {counts.shape[1]:,} trigrams; a window uses {np.mean(counts > 0):.1%} of them")

## Which trigrams drive the distance? (option C)

The Manhattan distance between two windows is a sum over trigrams of |count difference|. Adding that
up over every pair of windows gives each trigram's share of all the distance in the data.
Sorting a column makes the pairwise sum cheap: the k-th smallest of n values appears with a plus sign
k−1 times and a minus sign n−k times.

In [ ]:
def pairwise_abs_sum(column_counts: np.ndarray) -> np.ndarray:
    """Per column: the sum of |x_i - x_j| over all pairs of rows."""
    n = column_counts.shape[0]
    weights = 2 * np.arange(1, n + 1) - n - 1
    return weights @ np.sort(column_counts, axis=0)


contribution = pairwise_abs_sum(counts)
readback = pd.DataFrame({
    "trigram": [repr(t) for t in trigrams],
    "share_of_distance": contribution / contribution.sum(),
    "mean_count": counts.mean(axis=0),
    "windows_with_it": (counts > 0).mean(axis=0),
}).sort_values("share_of_distance", ascending=False).reset_index(drop=True)

cumulative = readback.share_of_distance.cumsum()
for top in (10, 50, 200):
    print(f"top {top:>3} trigrams: {cumulative.iloc[top - 1]:.0%} of all distance")
print(f"trigrams needed for half of all distance: {int((cumulative < 0.5).sum()) + 1:,} of {len(readback):,}")
readback.head(25)

In [ ]:
rare = readback[readback.windows_with_it < 3 / len(windows)]
print(f"trigrams in fewer than 3 windows: {len(rare):,} ({len(rare) / len(readback):.0%} of all), "
      f"together {rare.share_of_distance.sum():.0%} of all distance")